# 상미분방정식의 한 걸음과 누적 오차

Euler와 Runge–Kutta 방법의 첫 갱신을 이해하고 같은 초기조건에서 결과를 비교

코드 셀을 위에서 아래로 실행. 문제의 수치를 먼저 확인한 뒤 계산 결과를 비교

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## 같은 미분방정식, 다른 갱신식

y′=−2x³+12x²−20x+8.5, y(0)=1이다. h=0.5로 x=4까지 계산하시오. 먼저 Euler 방법으로 y(0.5)를 손계산하시오.

In [ ]:
def solve_ode(f,x0,y0,end,h,method):
    n=int(round((end-x0)/h))
    xs=np.linspace(x0,end,n+1)
    ys=[y0]
    for x in xs[:-1]:
        y=ys[-1];k1=f(x,y)
        if method=='Euler':
            slope=k1
        elif method=='Heun':
            slope=(k1+f(x+h,y+h*k1))/2
        elif method=='Midpoint':
            slope=f(x+h/2,y+h*k1/2)
        elif method=='Ralston':
            slope=k1/3+2*f(x+3*h/4,y+3*h*k1/4)/3
        else:
            k2=f(x+h/2,y+h*k1/2)
            k3=f(x+h/2,y+h*k2/2)
            k4=f(x+h,y+h*k3)
            slope=(k1+2*k2+2*k3+k4)/6
        ys.append(y+h*slope)
    return xs,np.array(ys)

f=lambda x,y:-2*x**3+12*x**2-20*x+8.5
exact=lambda x:-0.5*x**4+4*x**3-10*x*x+8.5*x+1
rows=[]
for method in ['Euler','Heun','Midpoint','Ralston','RK4']:
    xs,ys=solve_ode(f,0,1,4,0.5,method)
    rows.append([method,ys[1],ys[-1],np.max(np.abs(ys-exact(xs)))])
display(pd.DataFrame(rows,columns=['방법','y(0.5)','y(4)','최대 절대오차']))

Ralston 방법은 내부 평가점 x+3h/4, 가중치 1/3·2/3 사용. 첫 내부 y는 4.1875.

Euler의 h를 0.5에서 0.25로 줄여 같은 위치의 오차를 비교하시오.

In [ ]:
for h in [0.5,0.25]:
    xs,ys=solve_ode(f,0,1,4,h,"Euler")
    plt.plot(xs,ys,"o-",label=f"Euler h={h}")
    print(h,"x=4의 오차:",abs(ys[-1]-exact(4)))
grid=np.linspace(0,4,200);plt.plot(grid,exact(grid),"k",label="Exact")
plt.xlabel("x");plt.ylabel("y");plt.legend();plt.show()

## y에 따라 기울기가 달라지는 경우

y′=4e⁰·⁸ˣ−0.5y, y(0)=2이다. h=1로 x=4까지 Euler·Heun·RK4를 적용하시오.

In [ ]:
f2=lambda x,y:4*np.exp(0.8*x)-0.5*y
for method in ["Euler","Heun","RK4"]:
    xs,ys=solve_ode(f2,0,2,4,1,method)
    print(method,ys)
    plt.plot(xs,ys,"o-",label=method)
plt.xlabel("x");plt.ylabel("y");plt.legend();plt.show()

각 방법이 한 구간의 기울기를 어떻게 구하는지 갱신식과 연결하여 설명하시오.